# Clase 8 — VWAP Volume Baselines

**Prerequisito:** L7 LOB Modeling Examples  
**Duración demo:** ~15 min

---

En L7 predijimos la **dirección del precio** snapshot a snapshot — una señal micro y ruidosa con 55.5% de accuracy.  
Hoy cambiamos de escala completamente.

> **Pregunta central:** Tienes que comprar 10 BTC en las próximas 4 horas sin mover el mercado. ¿Cómo distribuyes las órdenes?

La respuesta se llama **VWAP** — Volume Weighted Average Price — y para construirlo necesitas predecir el **perfil de volumen intradiario**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('dark_background')
plt.rcParams.update({'font.family': 'monospace', 'axes.facecolor': '#18181b',
                     'figure.facecolor': '#09090b', 'axes.edgecolor': '#27272a',
                     'grid.color': '#27272a', 'text.color': '#e4e4e7'})

CYAN   = '#22d3ee'
GREEN  = '#4ade80'
RED    = '#f87171'
AMBER  = '#f59e0b'
PURPLE = '#a78bfa'
MUTED  = '#a1a1aa'

TOTAL_QTY = 10.0  # BTC a ejecutar

In [ ]:
df = pd.read_csv('data/btc_volume_intraday.csv', parse_dates=['datetime'])

print(f'Datos: {len(df):,} filas')
print(f'Días:  {df["date"].nunique()} (del {df["date"].min()} al {df["date"].max()})')
print(f'Intervalos/día: {df["interval_idx"].nunique()} (× 5 min = 24 h)')
print()
print(df[['datetime','weekday_name','interval_idx','volume','volume_normalized']].head(4).to_string())

## ¿Qué es el perfil intradiario?

La columna `volume_normalized` es la fracción del volumen diario que cayó en ese intervalo de 5 minutos.

- Si un intervalo histórico concentra el **2% del volumen diario**, tu schedule debe asignar el **2% de tu orden** a ese intervalo.
- Si tienes que comprar 10 BTC y ese intervalo vale 2%, envías **0.20 BTC** en ese slot.

Suma de todos los intervalos de un día: **exactamente 1.0** (por construcción).

In [ ]:
# Perfil medio — la forma U
mean_profile = df.groupby('interval_idx')['volume_normalized'].mean()

fig, ax = plt.subplots(figsize=(13, 4))

# 21 días individuales en gris muy tenue
for date, day_df in df.groupby('date'):
    profile = day_df.sort_values('interval_idx')['volume_normalized']
    ax.plot(profile.values, color=MUTED, linewidth=0.4, alpha=0.35)

# Media en cyan
ax.plot(mean_profile.values, color=CYAN, linewidth=2.5, label='Media (21 días)', zorder=5)

# Ticks horarios
xticks = [0, 72, 144, 216, 287]
xlabels = ['00:00', '06:00', '12:00', '18:00', '23:55']
ax.set_xticks(xticks)
ax.set_xticklabels(xlabels)
ax.set_xlabel('Hora (UTC)', color=MUTED)
ax.set_ylabel('Fracción volumen diario', color=MUTED)
ax.set_title('Perfil de volumen intradiario BTC — forma U', color=CYAN, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

ratio = mean_profile.iloc[0] / mean_profile.iloc[144]
ax.annotate(f'Open/midday: {ratio:.2f}×', xy=(0, mean_profile.iloc[0]),
            xytext=(20, mean_profile.iloc[0] * 1.1),
            color=AMBER, fontsize=9, arrowprops=dict(arrowstyle='->', color=AMBER))

plt.tight_layout()
plt.show()

print(f'Open (00:00):  {mean_profile.iloc[0]:.6f}')
print(f'Midday (12:00): {mean_profile.iloc[144]:.6f}')
print(f'Close (23:55): {mean_profile.iloc[287]:.6f}')
print(f'Ratio open/midday: {ratio:.4f}×')

In [ ]:
# Comparativa por día de la semana
dow_styles = {
    0: ('Monday',    RED,    2.0),
    2: ('Wednesday', MUTED,  1.2),
    4: ('Friday',    GREEN,  2.0),
}

fig, ax = plt.subplots(figsize=(13, 4))

for dow, (name, col, lw) in dow_styles.items():
    profile = df[df['weekday'] == dow].groupby('interval_idx')['volume_normalized'].mean()
    ax.plot(profile.values, color=col, linewidth=lw, label=name)

ax.plot(mean_profile.values, color=CYAN, linewidth=2, linestyle='--',
        label='All-days mean', alpha=0.8)

ax.set_xticks(xticks)
ax.set_xticklabels(xlabels)
ax.set_xlabel('Hora (UTC)', color=MUTED)
ax.set_ylabel('Fracción volumen diario', color=MUTED)
ax.set_title('Efecto día de la semana — lunes vs viernes', color=CYAN, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

mon_daily = df[df['weekday'] == 0].groupby('date')['volume'].sum().mean()
fri_daily = df[df['weekday'] == 4].groupby('date')['volume'].sum().mean()
print(f'Volumen medio lunes:   {mon_daily:,.0f} BTC/día')
print(f'Volumen medio viernes: {fri_daily:,.0f} BTC/día')
print(f'Ratio viernes/lunes:   {fri_daily/mon_daily:.2f}×')

## Las 4 predicciones baseline

| Baseline | Idea | Cuándo usar |
|----------|------|-------------|
| **mean_all** | Media de todos los días disponibles | Si no sabes el día |
| **median_all** | Mediana de todos los días | Más robusta a outliers |
| **mean_weekday** | Media de los lunes (si ejecutas un lunes) | Conoces el día de la semana |
| **median_weekday** | Mediana de los lunes | Lunes + robustez |

Objetivo: predecir qué **fracción del volumen diario** caerá en cada intervalo de 5 minutos.

In [ ]:
# Construir los 4 perfiles baseline
mean_all    = df.groupby('interval_idx')['volume_normalized'].mean()
median_all  = df.groupby('interval_idx')['volume_normalized'].median()
mean_monday = df[df['weekday'] == 0].groupby('interval_idx')['volume_normalized'].mean()
median_monday = df[df['weekday'] == 0].groupby('interval_idx')['volume_normalized'].median()

profiles = {
    'mean_all':      mean_all,
    'median_all':    median_all,
    'mean_monday':   mean_monday,
    'median_monday': median_monday,
}

print(f'{"Baseline":<20} {"sum":>10} {"max":>12} {"min":>12}')
print('-' * 56)
for name, p in profiles.items():
    print(f'{name:<20} {p.sum():>10.6f} {p.max():>12.8f} {p.min():>12.8f}')

## La función central: `build_vwap_schedule`

Dado un perfil de volumen y una cantidad total a ejecutar, el schedule es simplemente:

```
schedule[i] = total_qty × (profile[i] / sum(profile))
```

Tres líneas. Eso es VWAP básico. La dificultad no está en la función — **está en predecir bien el perfil**.

In [ ]:
def build_vwap_schedule(total_qty: float, volume_profile: pd.Series) -> pd.Series:
    """
    Distribuye total_qty sobre los intervalos del perfil,
    proporcional a volume_profile.
    
    Retorna: pd.Series con índice = interval_idx, valores en BTC.
    """
    normalized = volume_profile / volume_profile.sum()
    return total_qty * normalized

# Demo
schedule = build_vwap_schedule(TOTAL_QTY, mean_all)

print(f'Schedule construido: {len(schedule)} intervalos')
print(f'Total a ejecutar:    {schedule.sum():.6f} BTC')
print(f'Intervalo más activo: {schedule.argmax():3d} ({schedule.argmax()*5//60:02d}:{schedule.argmax()*5%60:02d} UTC) → {schedule.max():.4f} BTC')
print(f'Intervalo más quieto: {schedule.argmin():3d} ({schedule.argmin()*5//60:02d}:{schedule.argmin()*5%60:02d} UTC) → {schedule.min():.4f} BTC')

In [ ]:
# Visualizar los 4 schedules
colors = [CYAN, GREEN, RED, AMBER]
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True, sharey=True)
flat_level = TOTAL_QTY / 288

for ax, (name, prof), col in zip(axes.flat, profiles.items(), colors):
    sched = build_vwap_schedule(TOTAL_QTY, prof)
    ax.fill_between(range(288), sched.values, alpha=0.25, color=col)
    ax.plot(sched.values, color=col, linewidth=1.5)
    ax.axhline(flat_level, color=MUTED, linewidth=0.8, linestyle='--', label=f'Flat ({flat_level:.4f} BTC)')
    ax.set_title(name, color=col, fontweight='bold', fontsize=11)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xlabels, fontsize=8)
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)

fig.suptitle('VWAP Schedules — 10 BTC en 4 baselines (vs flat)', 
             color=CYAN, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Evaluación: ¿qué baseline predice mejor?

**Métrica:** RMSE entre el perfil predicho y los perfiles reales de cada día.

- Comparamos `pred_profile[i]` vs `actual_profile[i]` para cada intervalo de cada día en `eval_df`
- Menor RMSE = mejor predictor
- El RMSE opera sobre `volume_normalized` (fracciones ≈ 0.002 a 0.005)

In [ ]:
def rmse_profile(pred_profile: pd.Series, eval_df: pd.DataFrame) -> float:
    """
    RMSE entre el perfil predicho y los perfiles reales de cada día en eval_df.
    Opera sobre volume_normalized.
    """
    errors = []
    pred = pred_profile.values
    for date, day_df in eval_df.groupby('date'):
        actual = day_df.sort_values('interval_idx')['volume_normalized'].values
        errors.extend((actual - pred) ** 2)
    return float(np.sqrt(np.mean(errors)))

In [ ]:
# Evaluación global — todos los días
global_results = {name: rmse_profile(prof, df) for name, prof in profiles.items()}

print(f'Evaluación sobre TODOS los días (n=21):')
print(f'{"Baseline":<20} {"RMSE":>12}')
print('-' * 34)
for name, rmse in sorted(global_results.items(), key=lambda x: x[1]):
    best = ' ← mejor' if rmse == min(global_results.values()) else ''
    print(f'{name:<20} {rmse:>12.8f}{best}')

In [ ]:
# Evaluación Monday-only — 3 lunes
monday_eval = df[df['weekday'] == 0]
monday_results = {name: rmse_profile(prof, monday_eval) for name, prof in profiles.items()}

print(f'Evaluación solo LUNES (n=3):')
print(f'{"Baseline":<20} {"RMSE":>12}')
print('-' * 34)
for name, rmse in sorted(monday_results.items(), key=lambda x: x[1]):
    best = ' ← mejor para lunes' if rmse == min(monday_results.values()) else ''
    print(f'{name:<20} {rmse:>12.8f}{best}')

print()
improvement = (global_results['mean_all'] - monday_results['mean_monday']) / global_results['mean_all'] * 100
print(f'mean_monday mejora sobre mean_all en lunes: {improvement:.1f}%')

In [ ]:
# Comparativa visual — RMSE global vs lunes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

names = list(profiles.keys())
colors_bar = [CYAN, GREEN, RED, AMBER]
x = np.arange(len(names))

for ax, results, title in [
    (ax1, global_results, 'RMSE — todos los días (n=21)'),
    (ax2, monday_results, 'RMSE — solo lunes (n=3)'),
]:
    vals = [results[n] for n in names]
    bars = ax.bar(x, vals, color=colors_bar, alpha=0.8, width=0.6)
    best_idx = np.argmin(vals)
    bars[best_idx].set_edgecolor('white')
    bars[best_idx].set_linewidth(2)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
    ax.set_title(title, color=CYAN, fontweight='bold')
    ax.set_ylabel('RMSE', color=MUTED)
    ax.grid(axis='y', alpha=0.3)
    # Annotate best
    ax.annotate('← mejor', xy=(best_idx, vals[best_idx]),
                xytext=(best_idx + 0.4, vals[best_idx]),
                color=GREEN, fontsize=9,
                arrowprops=dict(arrowstyle='->', color=GREEN))

plt.tight_layout()
plt.show()

In [ ]:
# Helper de selección contextual
def select_best_profile(dow: int, all_profiles: dict) -> pd.Series:
    """
    Selecciona el mejor baseline según el día de la semana.
    Si hay un perfil específico para ese día, lo usa.
    Si no, cae en mean_all.
    """
    DOW_NAMES = {0: 'monday', 1: 'tuesday', 2: 'wednesday',
                 3: 'thursday', 4: 'friday', 5: 'saturday', 6: 'sunday'}
    key = f'mean_{DOW_NAMES[dow]}'
    return all_profiles.get(key, all_profiles['mean_all'])

# Demo: lunes (dow=0)
dow_today = 0
chosen = select_best_profile(dow_today, profiles)
schedule_today = build_vwap_schedule(TOTAL_QTY, chosen)

print(f'DOW {dow_today} (lunes): perfil seleccionado = mean_monday')
print(f'Órdenes a las 00:00: {schedule_today.iloc[0]:.4f} BTC')
print(f'Órdenes a las 12:00: {schedule_today.iloc[144]:.4f} BTC')
print(f'Open/midday ratio:   {schedule_today.iloc[0]/schedule_today.iloc[144]:.2f}×')

## Limitaciones de estos baselines

- **Estáticos:** el schedule se fija *antes* de empezar a ejecutar
- **Sin señal en tiempo real:** si el volumen real diverge del perfil predicho, el schedule no se adapta
- **Solo 21 días de historia:** con más datos el perfil sería más estable
- **Sin impacto de mercado dinámico:** no captura cómo *tu propia ejecución* mueve el mercado

---

## Puente a L9

En **L9** añadiremos una señal dinámica: los **últimos 5 minutos de volumen real** como corrección al baseline.  
El schedule dejará de ser estático — se actualizará con lo que ve el mercado.

La pregunta será: *¿cuánto mejora el RMSE si añadimos la señal reciente?*

In [ ]:
# Guardar artefactos para L9
mean_all.to_csv('data/mean_profile.csv', header=True)
print('Guardado: data/mean_profile.csv (L9 lo cargará directamente)')
print(f'Shape: {mean_profile.shape} — 288 intervalos, fracción del volumen diario')